# fig1_new_region - Part 8/8\n\n自动拆分版本（按步骤执行）。\n包含统一 bootstrap 和共享模块导入。\n

In [ ]:
# AUTO_BOOTSTRAP_V2
from pathlib import Path
import sys
import os
import builtins
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "projects").exists():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "projects").exists():
            ROOT = p
            break

NB_PATH = Path.cwd()
if "clone_motif" in str(NB_PATH):
    PROJECT_DIR = ROOT / "projects" / "clone_motif"
else:
    PROJECT_DIR = ROOT / "projects" / "our_multiregion_motif"

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
OUT_FIG = PROJECT_DIR / "outputs" / "figures"
OUT_TABLE = PROJECT_DIR / "outputs" / "tables"
OUT_ARCH = PROJECT_DIR / "outputs" / "archives"

for d in [DATA_PROCESSED, OUT_FIG, OUT_TABLE, OUT_ARCH]:
    d.mkdir(parents=True, exist_ok=True)

SHARED_SRC = ROOT / "projects" / "shared" / "src"
if str(SHARED_SRC) not in sys.path:
    sys.path.append(str(SHARED_SRC))

from motif_common import combination, indices_for_region, union_indices_for_regions, p_to_star, format_p_decimal_3sig, sort_by_order, truncate_colormap

READ_EXT = {".csv", ".json", ".npy", ".pkl", ".xlsx"}
FIG_EXT = {".svg", ".png", ".pdf"}
TABLE_EXT = {".csv", ".xlsx"}


def _as_path(x):
    return Path(x) if isinstance(x, (str, os.PathLike)) else x


def resolve_read_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute() or p.exists():
        return str(p)
    if p.suffix.lower() in READ_EXT:
        for c in [DATA_RAW / p.name, ROOT / p.name]:
            if c.exists():
                return str(c)
    return str(p)


def resolve_write_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    ext = p.suffix.lower()
    if ext in FIG_EXT:
        out = OUT_FIG / p.name
    elif ext in TABLE_EXT:
        out = OUT_TABLE / p.name
    elif ext == ".zip":
        out = OUT_ARCH / p.name
    elif ext == ".npy":
        out = DATA_PROCESSED / p.name
    else:
        out = PROJECT_DIR / p
    out.parent.mkdir(parents=True, exist_ok=True)
    return str(out)

if not hasattr(builtins, "_orig_open_codex"):
    builtins._orig_open_codex = builtins.open


def _open_patch(file, mode="r", *args, **kwargs):
    if isinstance(file, (str, os.PathLike)):
        if any(m in mode for m in ["r", "a"]):
            file = resolve_read_path(file)
        if any(m in mode for m in ["w", "a", "x"]):
            file = resolve_write_path(file)
    return builtins._orig_open_codex(file, mode, *args, **kwargs)


builtins.open = _open_patch

if not hasattr(np, "_orig_load_codex"):
    np._orig_load_codex = np.load
np.load = lambda file, *a, **k: np._orig_load_codex(resolve_read_path(file), *a, **k)

if not hasattr(np, "_orig_save_codex"):
    np._orig_save_codex = np.save
np.save = lambda file, arr, *a, **k: np._orig_save_codex(resolve_write_path(file), arr, *a, **k)

if not hasattr(pd, "_orig_read_csv_codex"):
    pd._orig_read_csv_codex = pd.read_csv
pd.read_csv = lambda f, *a, **k: pd._orig_read_csv_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd, "_orig_read_excel_codex"):
    pd._orig_read_excel_codex = pd.read_excel
pd.read_excel = lambda f, *a, **k: pd._orig_read_excel_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd.DataFrame, "_orig_to_csv_codex"):
    pd.DataFrame._orig_to_csv_codex = pd.DataFrame.to_csv


def _to_csv_patch(self, path_or_buf=None, *args, **kwargs):
    if isinstance(path_or_buf, (str, os.PathLike)):
        path_or_buf = resolve_write_path(path_or_buf)
    return pd.DataFrame._orig_to_csv_codex(self, path_or_buf, *args, **kwargs)


pd.DataFrame.to_csv = _to_csv_patch

if not hasattr(pd.DataFrame, "_orig_to_excel_codex"):
    pd.DataFrame._orig_to_excel_codex = pd.DataFrame.to_excel


def _to_excel_patch(self, excel_writer, *args, **kwargs):
    if isinstance(excel_writer, (str, os.PathLike)):
        excel_writer = resolve_write_path(excel_writer)
    return pd.DataFrame._orig_to_excel_codex(self, excel_writer, *args, **kwargs)


pd.DataFrame.to_excel = _to_excel_patch

try:
    import matplotlib.pyplot as plt
    if not hasattr(plt, "_orig_savefig_codex"):
        plt._orig_savefig_codex = plt.savefig
    plt.savefig = lambda f, *a, **k: plt._orig_savefig_codex(resolve_write_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)
except Exception:
    pass

print(f"[bootstrap] project={PROJECT_DIR.name} data={DATA_RAW}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 要比较的区域 =====
region_list = ['FRP']   # 你要看的区域
ref_region = 'MOp'      # 参考区域

# 统一收集（去掉不存在的）
all_regions = region_list + [ref_region]
all_regions = [r for r in all_regions if r in motif_results]

thr_to_use = 5  # 对应 motif_results[region][thr_to_use]，按你实际用的改

motif_ids = np.arange(1, 14)
motif_labels = [f"M{i}" for i in motif_ids]

# ===== 收集 NZ 以及 N / density =====
nz_dict = {}
info_dict = {}  # N, density

for region in all_regions:
    d_thr = motif_results[region].get(thr_to_use, None)
    if d_thr is None:
        print(f"[WARN] {region} 没有 thr={thr_to_use} 的结果，跳过")
        continue

    NZ = np.asarray(d_thr["NZ"], dtype=float)  # (13,)

    nz_dict[region] = NZ
    info_dict[region] = (d_thr["N"], d_thr["density"])

plot_regions = list(nz_dict.keys())
print("参与比较的区域:", plot_regions)

# ====== 1. 原始图像：NZ-score（包含 MOp） ======
fig, ax1 = plt.subplots(1, 1, figsize=(8, 4))

for region in plot_regions:
    NZ = nz_dict[region]
    N, dens = info_dict[region]
    label = f"{region} (N={N}, ρ={dens:.3f})"
    if region == ref_region:
        ax1.plot(motif_ids, NZ, marker='o', linewidth=2.5, label=label)
    else:
        ax1.plot(motif_ids, NZ, marker='o', linewidth=1.5, label=label)

ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_xlabel("Motif ID")
ax1.set_xticks(motif_ids)
ax1.set_xticklabels(motif_labels)
ax1.set_ylabel("NZ-score")
ax1.set_title(f"NZ-score comparison (thr={thr_to_use})")
ax1.legend(fontsize=8, ncol=2)
ax1.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ====== 2. 和 MOp 的差值：NZ 差值 ======
if ref_region not in nz_dict:
    print(f"[ERROR] 参考区域 {ref_region} 不在 nz_dict 里，无法做差值。")
else:
    nz_ref = nz_dict[ref_region]

    fig, ax2 = plt.subplots(1, 1, figsize=(8, 4))

    for region in plot_regions:
        if region == ref_region:
            continue  # MOp 本身差值是 0，不必画
        NZ = nz_dict[region]
        NZ_diff = NZ - nz_ref
        N, dens = info_dict[region]
        label = f"{region}−{ref_region} (N={N}, ρ={dens:.3f})"
        ax2.plot(motif_ids, NZ_diff, marker='o', linewidth=1.8, label=label)

    ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax2.set_xlabel("Motif ID")
    ax2.set_xticks(motif_ids)
    ax2.set_xticklabels(motif_labels)
    ax2.set_ylabel("Δ NZ (region − MOp)")
    ax2.set_title(f"NZ-score difference vs {ref_region} (thr={thr_to_use})")
    ax2.legend(fontsize=8, ncol=2)
    ax2.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()